# Meta-Learning for Algorithm Selection from Dataset Meta-Features

**Core analysis notebook** — reproduces every numeric result in the study.

This notebook orchestrates the modules in `src/`. Each step calls a
function that also writes its output to `results/`, so the notebook is both
the narrative and the reproducibility harness.

**Pipeline**
1. Setup & imports
2. Load / download the 20 datasets
3. Extract meta-features
4. Train 5 algorithms (5-fold stratified CV)
5. Build the performance matrix
6. Rank algorithms per dataset
7. Train the meta-learner
8. Statistical tests (Friedman + Nemenyi, Wilcoxon)
9. Export results for the visualisation and XML tracks

> Run top-to-bottom. Steps 2 and 4 download data and train models; later
> runs reuse cached artefacts, so re-execution is fast.

## 1. Setup & imports

In [1]:
import sys
from pathlib import Path

# Make the repository root importable so `import src...` works from notebooks/.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from src import data_loader, meta_features, models, ranking, stats, meta_learner

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
results_dir = REPO_ROOT / "results"
print("Repository root:", REPO_ROOT)
print("Modules loaded: data_loader, meta_features, models, ranking, stats, meta_learner")

Repository root: C:\Users\USER\desktop\XML_Project\meta-learning-algorithm-selection
Modules loaded: data_loader, meta_features, models, ranking, stats, meta_learner


## 2. Load / download the 20 datasets

`data_loader.select_and_download()` fetches a balanced set of PMLB datasets
(8 binary, 7 multiclass, 5 regression) and writes them to `data/processed/`.
If they already exist we just load the index.

In [2]:
try:
    index = data_loader.load_index()
    print(f"Using cached datasets ({len(index)} in index).")
except FileNotFoundError:
    index = data_loader.select_and_download()

index

Using cached datasets (20 in index).


,dataset,task,n_instances,n_features,n_classes,target_column,source,file
0,GAMETES_Epistasis_2_Way_20atts_0.1H_EDM_1_1,binary,1600,20,2,target,PMLB,data/processed/GAMETES_Epistasis_2_Way_20atts_...
1,GAMETES_Epistasis_2_Way_20atts_0.4H_EDM_1_1,binary,1600,20,2,target,PMLB,data/processed/GAMETES_Epistasis_2_Way_20atts_...
2,GAMETES_Epistasis_3_Way_20atts_0.2H_EDM_1_1,binary,1600,20,2,target,PMLB,data/processed/GAMETES_Epistasis_3_Way_20atts_...
3,GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_...,binary,1600,20,2,target,PMLB,data/processed/GAMETES_Heterogeneity_20atts_16...
4,GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_...,binary,1600,20,2,target,PMLB,data/processed/GAMETES_Heterogeneity_20atts_16...
5,analcatdata_boxing1,binary,120,3,2,target,PMLB,data/processed/analcatdata_boxing1.csv
6,analcatdata_boxing2,binary,132,3,2,target,PMLB,data/processed/analcatdata_boxing2.csv
7,analcatdata_creditscore,binary,100,6,2,target,PMLB,data/processed/analcatdata_creditscore.csv
8,analcatdata_dmft,multiclass,797,4,6,target,PMLB,data/processed/analcatdata_dmft.csv
9,analcatdata_germangss,multiclass,400,5,4,target,PMLB,data/processed/analcatdata_germangss.csv


## 3. Extract meta-features

For each dataset we compute 13 meta-features describing size, dimensionality,
class structure and simple statistical properties. This matrix is the input
to the meta-learner in step 7.

In [3]:
meta_matrix = meta_features.build_meta_feature_matrix()
meta_matrix

  meta-features: GAMETES_Epistasis_2_Way_20atts_0.1H_EDM_1_1 (binary)
  meta-features: GAMETES_Epistasis_2_Way_20atts_0.4H_EDM_1_1 (binary)
  meta-features: GAMETES_Epistasis_3_Way_20atts_0.2H_EDM_1_1 (binary)
  meta-features: GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_50_EDM_2_001 (binary)
  meta-features: GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_75_EDM_2_001 (binary)
  meta-features: analcatdata_boxing1 (binary)
  meta-features: analcatdata_boxing2 (binary)
  meta-features: analcatdata_creditscore (binary)
  meta-features: analcatdata_dmft (multiclass)
  meta-features: analcatdata_germangss (multiclass)
  meta-features: balance_scale (multiclass)
  meta-features: calendarDOW (multiclass)
  meta-features: car_evaluation (multiclass)
  meta-features: cars (multiclass)
  meta-features: cloud (multiclass)
  meta-features: 1027_ESL (regression)
  meta-features: 1028_SWD (regression)
  meta-features: 1029_LEV (regression)
  meta-features: 1030_ERA (regression)
  meta-features: 210_

,task,n_instances,n_features,log_n_instances,log_n_features,feature_to_instance_ratio,n_classes,class_imbalance_ratio,normalised_class_entropy,mean_feature_std,mean_abs_coef_variation,mean_skewness,mean_kurtosis,mean_abs_feature_correlation
dataset,,,,,,,,,,,,,,
GAMETES_Epistasis_2_Way_20atts_0.1H_EDM_1_1,binary,1600.0,20.0,3.204120,1.301030,0.012500,2.0,1.000000,1.000000,0.578571,1.393807,0.962878,0.473353,0.020243
GAMETES_Epistasis_2_Way_20atts_0.4H_EDM_1_1,binary,1600.0,20.0,3.204120,1.301030,0.012500,2.0,1.000000,1.000000,0.548823,1.861129,1.454625,4.077101,0.021586
GAMETES_Epistasis_3_Way_20atts_0.2H_EDM_1_1,binary,1600.0,20.0,3.204120,1.301030,0.012500,2.0,1.000000,1.000000,0.549796,1.565391,1.198508,1.013058,0.020139
GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_50_EDM_2_001,binary,1600.0,20.0,3.204120,1.301030,0.012500,2.0,1.000000,1.000000,0.527997,1.802731,1.428780,2.863247,0.020228
GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_75_EDM_2_001,binary,1600.0,20.0,3.204120,1.301030,0.012500,2.0,1.000000,1.000000,0.576352,1.584818,1.148813,2.690301,0.020326
analcatdata_boxing1,binary,120.0,3.0,2.079181,0.477121,0.025000,2.0,1.857143,0.934068,2.260864,0.898965,0.290957,-1.226374,0.063311
analcatdata_boxing2,binary,132.0,3.0,2.120574,0.477121,0.022727,2.0,1.163934,0.995856,2.353231,0.932178,0.340207,-1.131706,0.172133
analcatdata_creditscore,binary,100.0,6.0,2.000000,0.778151,0.060000,2.0,2.703704,0.841465,50.647164,1.792858,2.410571,8.458371,0.149203
analcatdata_dmft,multiclass,797.0,4.0,2.901458,0.602060,0.005019,6.0,1.260163,0.998181,1.372348,0.800094,0.051826,-1.150275,0.159096


## 4-5. Train 5 algorithms (5-fold CV) & build the performance matrix

`models.run_all()` evaluates Random Forest, Gradient Boosting, SVM, k-NN and
a neural network with 5-fold stratified cross-validation on every dataset
(accuracy for classification, R2 for regression). It writes the long results
(one row per dataset x algorithm x fold) and the wide performance matrix.

> Heaviest step (500 model fits). Cached: delete
> `results/performance_matrix.csv` to force a full re-run.

In [4]:
if (results_dir / "performance_matrix.csv").exists():
    print("Using cached CV results.")
else:
    models.run_all()

performance_matrix = models.load_performance_matrix()
performance_matrix.round(4)

Using cached CV results.


,RandomForest,GradientBoosting,SVM,kNN,NeuralNetwork
dataset,,,,,
GAMETES_Epistasis_2_Way_20atts_0.1H_EDM_1_1,0.6031,0.5981,0.5788,0.5381,0.5550
GAMETES_Epistasis_2_Way_20atts_0.4H_EDM_1_1,0.6906,0.6606,0.6694,0.5994,0.6631
GAMETES_Epistasis_3_Way_20atts_0.2H_EDM_1_1,0.5419,0.5106,0.5212,0.5387,0.5275
GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_50_EDM_2_001,0.6319,0.6725,0.6338,0.5488,0.6169
GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_75_EDM_2_001,0.6481,0.6875,0.6631,0.5894,0.6238
analcatdata_boxing1,0.7583,0.8250,0.7167,0.6583,0.7917
analcatdata_boxing2,0.6829,0.7043,0.6601,0.6977,0.6980
analcatdata_creditscore,0.9900,0.9700,0.7900,0.8000,0.8900
analcatdata_dmft,0.1744,0.2209,0.2246,0.1794,0.1932


## 6. Rank algorithms per dataset

Ranks (1 = best) are scale-free, so they are comparable across accuracy and
R2 tasks. The best algorithm per dataset becomes the meta-learner's target.

In [5]:
rankings = ranking.build_rankings()

print("Average ranks (lower is better):")
display(rankings["average_ranks"].to_frame())
print("Best algorithm per dataset:")
display(rankings["best_algorithm"].value_counts().to_frame("n_datasets"))

Average ranks (lower is better):
GradientBoosting    2.00
NeuralNetwork       2.40
RandomForest        2.90
SVM                 3.45
kNN                 4.25

Best-algorithm counts:
best_algorithm
GradientBoosting    9
NeuralNetwork       5
RandomForest        4
SVM                 2

Ranking artefacts -> C:\Users\USER\Desktop\XML_Project\meta-learning-algorithm-selection\results
Average ranks (lower is better):


,average_rank
GradientBoosting,2.00
NeuralNetwork,2.40
RandomForest,2.90
SVM,3.45
kNN,4.25


Best algorithm per dataset:


,n_datasets
best_algorithm,
GradientBoosting,9
NeuralNetwork,5
RandomForest,4
SVM,2


## 7. Train the meta-learner

The empirical centrepiece: a Random Forest that predicts the best algorithm
from meta-features alone, evaluated with leave-one-out CV (20 datasets)
against a majority-class baseline. Also yields meta-feature importance
(Table 2).

In [6]:
meta_summary = meta_learner.run_meta_learner()

importance = pd.read_csv(results_dir / "meta_feature_importance.csv")
print("Meta-feature importance (Table 2):")
display(importance)

Meta-learner (predict best algorithm from meta-features)
  LOO-CV accuracy   : 0.350  (7/20 correct)
  Baseline (majority 'GradientBoosting'): 0.450
  Lift over baseline: -0.100

Top meta-features (importance):
    mean_abs_coef_variation          0.1258
    mean_skewness                    0.1219
    feature_to_instance_ratio        0.1097
    mean_kurtosis                    0.0979
    mean_feature_std                 0.0939
    mean_abs_feature_correlation     0.0798

Meta-learner artefacts -> C:\Users\USER\Desktop\XML_Project\meta-learning-algorithm-selection\results
Meta-feature importance (Table 2):


,meta_feature,importance
0,mean_abs_coef_variation,0.125782
1,mean_skewness,0.121894
2,feature_to_instance_ratio,0.109724
3,mean_kurtosis,0.097939
4,mean_feature_std,0.093895
5,mean_abs_feature_correlation,0.079785
6,n_features,0.068422
7,n_instances,0.059506
8,log_n_instances,0.052092
9,n_classes,0.050373


## 8. Statistical tests

- **Friedman**: do the algorithms differ in rank across datasets?
- **Nemenyi**: which pairs differ (Critical Difference -> Figure 1)?
- **Wilcoxon signed-rank**: best algorithm vs each competitor on raw scores.

In [7]:
test_summary = stats.run_all_tests()

print("Nemenyi pairwise comparisons:")
display(pd.read_csv(results_dir / "nemenyi_pairwise.csv"))
print("Wilcoxon vs best algorithm:")
display(pd.read_csv(results_dir / "wilcoxon_vs_best.csv"))

Friedman test:
  chi^2 = 25.0800, p = 4.848e-05 (N=20, k=5)
  significant -> algorithms differ at alpha=0.05

Nemenyi critical difference (alpha=0.05): 1.3639
  3 of 10 pairs significantly different
    GradientBoosting vs SVM: diff=1.450
    GradientBoosting vs kNN: diff=2.250
    kNN vs NeuralNetwork: diff=1.850

Wilcoxon signed-rank vs best (GradientBoosting):
  * vs RandomForest     p=0.00169
  * vs SVM              p=0.002712
  * vs kNN              p=1.907e-05
  * vs NeuralNetwork    p=0.02958

Statistical artefacts -> C:\Users\USER\Desktop\XML_Project\meta-learning-algorithm-selection\results
Nemenyi pairwise comparisons:


,algorithm_a,algorithm_b,rank_diff,significant
0,RandomForest,GradientBoosting,0.90,False
1,RandomForest,SVM,0.55,False
2,RandomForest,kNN,1.35,False
3,RandomForest,NeuralNetwork,0.50,False
4,GradientBoosting,SVM,1.45,True
5,GradientBoosting,kNN,2.25,True
6,GradientBoosting,NeuralNetwork,0.40,False
7,SVM,kNN,0.80,False
8,SVM,NeuralNetwork,1.05,False
9,kNN,NeuralNetwork,1.85,True


Wilcoxon vs best algorithm:


,best_algorithm,compared_with,statistic,p_value,best_mean_score,other_mean_score
0,GradientBoosting,RandomForest,25.0,0.001690,0.676346,0.629678
1,GradientBoosting,SVM,28.0,0.002712,0.676346,0.595108
2,GradientBoosting,kNN,5.0,0.000019,0.676346,0.566331
3,GradientBoosting,NeuralNetwork,47.0,0.029575,0.676346,0.620249


## 9. Results export summary

All numeric artefacts are now in `results/`. These are the inputs for the
downstream tracks: the visualisation figures/tables, the XSD/XML modelling,
and the manuscript.

In [8]:
print("Artefacts written to results/:")
for path in sorted(results_dir.glob("*")):
    if path.is_file():
        print(f"  {path.name:32s} {path.stat().st_size:>8,} bytes")

print("\nHeadline findings")
print(f"  Friedman p-value      : {test_summary['friedman']['p_value']:.2e}")
print(f"  Nemenyi CD (alpha=.05): {test_summary['nemenyi']['critical_difference']:.3f}")
print(f"  Best algorithm overall: {rankings['average_ranks'].idxmin()} "
      f"(avg rank {rankings['average_ranks'].min():.2f})")
print(f"  Meta-learner LOO acc  : {meta_summary['loo_accuracy']:.3f} "
      f"(baseline {meta_summary['baseline_accuracy']:.3f})")

Artefacts written to results/:
  average_ranks.csv                      94 bytes
  best_algorithm.csv                    778 bytes
  cv_results_long.csv                40,965 bytes
  meta_feature_importance.csv           565 bytes
  meta_features.csv                   4,406 bytes
  meta_learner_predictions.csv        1,192 bytes
  meta_learner_summary.json             655 bytes
  nemenyi_pairwise.csv                  455 bytes
  normalised_scores.csv               1,822 bytes
  performance_matrix.csv              2,068 bytes
  ranks.csv                             932 bytes
  statistical_tests.json              3,065 bytes
  training_log.txt                    4,646 bytes
  wilcoxon_vs_best.csv                  442 bytes

Headline findings
  Friedman p-value      : 4.85e-05
  Nemenyi CD (alpha=.05): 1.364
  Best algorithm overall: GradientBoosting (avg rank 2.00)
  Meta-learner LOO acc  : 0.350 (baseline 0.450)
